<center><h1>Predictive Analytics (QBUS2820)</h1></center>
<center><h2>Tutorial 7: Variable Selection</h2></center>
<br>

In this tutorial, we will study how to implement the variable selection in Python. 

This notebook relies on the following imports and settings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_context('notebook') 
sns.set_style('ticks')

### Data

Our analysis will be based on a customer acquisition and profitability dataset. The detailed data description can be found [here](https://examples.rpkg.net/packages/SMCRM/reference/customerAcquisition.ob).


In [2]:
data = pd.read_excel('CustomerAcquisition.xls', index_col=[0])
data.head()

,Acquisition,First_Purchase,CLV,Duration,Censor,Acq_Expense,Acq_Expense_SQ,Industry,Revenue,Employees,Ret_Expense,Ret_Expense_SQ,Crossbuy,Frequency,Frequency_SQ
Customer,,,,,,,,,,,,,,,
1,1,433.64,0.0000,384,0,760.36,578147.33,1,30.16,1240,2309.79,5335129.84,5,2,4
2,0,0.00,0.0000,0,0,147.70,21815.29,1,39.80,166,0.00,0.00,0,0,0
3,0,0.00,0.0000,0,0,252.56,63786.55,1,54.93,1016,0.00,0.00,0,0,0
4,1,225.84,5.7316,730,1,609.73,371770.67,1,45.83,122,2192.59,4807450.91,2,12,144
5,1,363.04,0.0000,579,0,672.36,452067.97,1,69.03,313,801.14,641825.30,4,7,49


The objective of this analysis is to predict customer profitability, which involves both classification and regression tasks. Classification methods are used to predict whether a customer will be acquired and retained, while regression methods estimate the profitability of the acquired and retained customers to the company.  Here, we focus on the regression task. 

The response variable for this tutorial is the customer lifetime value (``CLV``), a measure of profitability, of customers who was still a customer at the end of the observation window (730 days). These customers are labeled with a value of one in the ``Censor`` variable.

In [3]:
data = data[data['Censor']==1]

The predictors are:
- ``First_Purchase``: Dollar value of the ﬁrst purchase (0 if the customer was not acquired).
- ``Acq_Expense``: Dollars spent on marketing efforts to try and acquire that prospect.
- ``Acq_Expense_SQ``: Square of dollars spent on marketing efforts to try and acquire that prospect.
- ``Industry``: 1 if the prospect is in the B2B industry, 0 otherwise.
- ``Revenue``: Annual sales revenue of the prospect’s ﬁrm (in millions of dollars).
- ``Employees``: Number of employees in the prospect’s ﬁrm.
- ``Ret_Expense``: Dollars spent on marketing efforts to try and retain that customer.
- ``Ret_Expense_SQ``: Square of dollars spent on marketing efforts to try and retain that customer.
- ``Crossbuy``: The number of categories the customer has purchased.
- ``Frequency``: The number of times the customer purchased during the observation window.
- ``Frequency_SQ``: The square of the number of times the customer purchased during the observation window.

Below, we record the name of the response and predictor variables for subsequent use. This is efficient since most of the code is completely automated from here onwards, allowing us to quickly implement the same type of analysis for different data. 

In [4]:
response = ['CLV']
exclude = response + ['Acquisition', 'Duration', 'Censor'] # list of variables who are NOT predictors
predictors=[x for x in list(data.columns) if x not in exclude] # building a list of predictors
data=data[response+predictors] # excluding variables which we are not going to use
data.head()

,CLV,First_Purchase,Acq_Expense,Acq_Expense_SQ,Industry,Revenue,Employees,Ret_Expense,Ret_Expense_SQ,Crossbuy,Frequency,Frequency_SQ
Customer,,,,,,,,,,,,
4,5.7316,225.84,609.73,371770.67,1,45.83,122,2192.59,4807450.91,2,12,144
9,6.9161,599.30,452.35,204620.52,1,17.98,1782,1340.75,1797610.56,1,11,121
10,6.0839,271.18,786.72,618928.36,1,38.91,539,2265.93,5134438.76,1,14,196
14,8.7544,678.06,472.13,222906.74,0,58.91,1507,2430.31,5906406.70,5,2,4
16,6.5147,178.28,610.91,373211.03,1,33.78,41,2071.53,4291236.54,6,8,64


List of the potential predictors:

In [5]:
predictors

['First_Purchase',
 'Acq_Expense',
 'Acq_Expense_SQ',
 'Industry',
 'Revenue',
 'Employees',
 'Ret_Expense',
 'Ret_Expense_SQ',
 'Crossbuy',
 'Frequency',
 'Frequency_SQ']

We are ready to split the data into training and test sets. 

In [6]:
train = data.sample(frac=0.5, random_state=1)
test = data[data.index.isin(train.index)==False].copy()

### Standardizing the Predictors

Regularization methods require standardizing the predictors by subtracting the mean (centering to 0 mean) and dividing by the standard deviation (scaling to unit variance). It’s important to apply the same transformations to the test data using the mean and standard deviation derived from the **training data**.

Alternatively, you can use the [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler) from ``scikit-learn`` for this task. 

There is no need to standardize the response variable, as **Lasso, Ridge, and ElasticNet** can automatically calculate the intercept term (``fit_intercept=True`` by default). Therefore, predictions will remain on the original scale of the response variable.

In [7]:
mu=train[predictors].mean() # mean for each feature
sigma=train[predictors].std() # std for each feature

In [8]:
mu.head()

First_Purchase       424.133824
Acq_Expense          563.333824
Acq_Expense_SQ    328524.992647
Industry               0.705882
Revenue               40.867500
dtype: float64

In [9]:
sigma.head()

First_Purchase       149.298535
Acq_Expense          106.521648
Acq_Expense_SQ    119095.465998
Industry               0.459033
Revenue               16.173683
dtype: float64

Standardize the predictors in both the training and test data. **Note that it’s crucial to use the mean and standard deviation calculated from the training data.**

In [10]:
train[predictors]=(train[predictors]-mu)/sigma
test[predictors]=(test[predictors]-mu)/sigma

Check the means of the predictors (in training data) after standardization. Observe that the data have been centered (i.e. having a zero mean).

In [11]:
train[predictors].mean().head()

First_Purchase    7.510332e-17
Acq_Expense      -3.020460e-16
Acq_Expense_SQ    2.171466e-16
Industry         -8.816477e-17
Revenue          -4.506199e-16
dtype: float64

Check the standard deviations of the predictors (in training data) after standardization.

In [12]:
train[predictors].std().head()

First_Purchase    1.0
Acq_Expense       1.0
Acq_Expense_SQ    1.0
Industry          1.0
Revenue           1.0
dtype: float64

### Variable Selection in OLS
We will demonstrate variable selection in OLS based on both the hypothesis testing and AIC/BIC criteria. 
Note that, technically, unlike regularization methods, OLS does not require us to standardize the data. But it is often more numerically stable to do so.

In [13]:
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from scipy import stats

x_with_intercept = sm.add_constant(train[predictors], prepend=True)
ols = sm.OLS(train[response],x_with_intercept)
est = ols.fit()
print(est.summary())


                            OLS Regression Results                            
Dep. Variable:                    CLV   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     32.51
Date:                Sun, 28 Jul 2024   Prob (F-statistic):           2.44e-20
Time:                        13:52:56   Log-Likelihood:                -26.423
No. Observations:                  68   AIC:                             76.85
Df Residuals:                      56   BIC:                             103.5
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              6.4117      0.048    134.

As the p-value for``Revenue`` is the largest (larger than 0.05), let's remove this predictor from the model.

In [14]:
predictors_new = ['First_Purchase',
 'Acq_Expense',
 'Acq_Expense_SQ',
 'Industry',
# 'Revenue',
 'Employees',
 'Ret_Expense',
 'Ret_Expense_SQ',
 'Crossbuy',
 'Frequency',
 'Frequency_SQ']

In [15]:
x_with_intercept = sm.add_constant(train[predictors_new], prepend=True)
ols = sm.OLS(train[response],x_with_intercept)
est = ols.fit()
print(est.summary())

                            OLS Regression Results                            
Dep. Variable:                    CLV   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     36.39
Date:                Sun, 28 Jul 2024   Prob (F-statistic):           3.94e-21
Time:                        13:52:56   Log-Likelihood:                -26.430
No. Observations:                  68   AIC:                             74.86
Df Residuals:                      57   BIC:                             99.27
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              6.4117      0.047    135.

With ``Revenue`` removed, both AIC and BIC got smaller. Now, the p-value of ``Acq_Expense`` is quite high, next, we try to remove this predictor.

In [16]:
predictors_new = ['First_Purchase',
# 'Acq_Expense',
 'Acq_Expense_SQ',
 'Industry',
# 'Revenue',
 'Employees',
 'Ret_Expense',
 'Ret_Expense_SQ',
 'Crossbuy',
 'Frequency',
 'Frequency_SQ']

In [17]:
x_with_intercept = sm.add_constant(train[predictors_new], prepend=True)
ols = sm.OLS(train[response],x_with_intercept)
est = ols.fit()
print(est.summary())

                            OLS Regression Results                            
Dep. Variable:                    CLV   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.843
Method:                 Least Squares   F-statistic:                     40.86
Date:                Sun, 28 Jul 2024   Prob (F-statistic):           7.00e-22
Time:                        13:52:56   Log-Likelihood:                -26.628
No. Observations:                  68   AIC:                             73.26
Df Residuals:                      58   BIC:                             95.45
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              6.4117      0.047    136.

Both AIC and BIC got smaller, hence we should remove ``Acq_Expense``. **Your task: try to remove ``Employees``**.  

### Regularization Methods

Let's find the Lasso regression coefficients with the shrinkage parameter ``alpha = 0.01``, for example.
See [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso) for how to use this function.

If the predictor data $X$ have not been standardized, you can set **normalize=True**, then $X$ will be standardized before Lasso regression.

In [18]:
from sklearn import linear_model
Lasso_reg = linear_model.Lasso(alpha=0.01)
Lasso_reg.fit(train[predictors], np.ravel(train[response]))
pd.DataFrame(Lasso_reg.coef_.round(3), index = predictors).T

,First_Purchase,Acq_Expense,Acq_Expense_SQ,Industry,Revenue,Employees,Ret_Expense,Ret_Expense_SQ,Crossbuy,Frequency,Frequency_SQ
0,0.488,-0.0,-0.301,0.254,0.043,0.0,1.201,-0.617,0.328,0.452,-0.174


In [19]:
from sklearn import linear_model
Lasso_reg = linear_model.Lasso(alpha=0.02)
Lasso_reg.fit(train[predictors], np.ravel(train[response]))
pd.DataFrame(Lasso_reg.coef_.round(3), index = predictors).T

,First_Purchase,Acq_Expense,Acq_Expense_SQ,Industry,Revenue,Employees,Ret_Expense,Ret_Expense_SQ,Crossbuy,Frequency,Frequency_SQ
0,0.465,-0.0,-0.302,0.238,0.044,0.0,0.571,-0.0,0.319,0.25,-0.0


With ``alpha=0.01``, two predictors ``Acq_Expense`` and ``Employees`` are removed. 
With ``alpha=0.02``, four predictors ``Acq_Expense``, ``Employees``, ``Ret_Expense_SQ`` and ``Frequency`` are removed. 

**How can we find optimal ``alpha``?** 

[LassoCV](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) function is the most convenient implementation with built in CV-based model selection for tuning parameter $\alpha$. 

In [20]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=10)
lasso.fit(train[predictors], np.ravel(train[response])) # the np.ravel is a necessary detail for compatibility

LassoCV(cv=10)

In [21]:
print("LASSO Lambda: {0}".format(lasso.alpha_))

LASSO Lambda: 0.0032143930090217892


In [22]:
pd.DataFrame(lasso.coef_.round(3), index = predictors).T

,First_Purchase,Acq_Expense,Acq_Expense_SQ,Industry,Revenue,Employees,Ret_Expense,Ret_Expense_SQ,Crossbuy,Frequency,Frequency_SQ
0,0.525,0.0,-0.288,0.264,0.037,-0.012,1.694,-1.1,0.331,0.729,-0.435


In [23]:
# Get the predictors with non-zero coefficients
selected_predictors = [predictors[i] for i, coef in enumerate(lasso.coef_) if coef != 0]

# Refit OLS model using only the selected predictors
x_with_intercept = sm.add_constant(train[selected_predictors], prepend=True)
ols = sm.OLS(train[response],x_with_intercept)
est = ols.fit()
print(est.summary())

                            OLS Regression Results                            
Dep. Variable:                    CLV   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                     36.14
Date:                Sun, 28 Jul 2024   Prob (F-statistic):           4.62e-21
Time:                        13:52:56   Log-Likelihood:                -26.626
No. Observations:                  68   AIC:                             75.25
Df Residuals:                      57   BIC:                             99.67
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              6.4117      0.047    135.

## Homework task: 
Implement the BIC method for selecting $\alpha$.